# Stage1 SOXX 日成交量参与度：来源与冗余预检

只读原 Stage1 Qlib Recorder 和已认证的 Futu published-affine Qlib 日线源。前一完整交易日 16:15 的数据对应下一交易日 09:29 决策；历史接收时刻仍是限制。本卡不读取 2023+ 收益、不拟合、不回测。

In [1]:
import torch
import json,hashlib
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.workflow import R
root=Path.cwd();card=root/'research/stage1_daily_volume_participation_source_20261022';plan=json.loads((card/'plan.json').read_text())
provider=root/plan['source_provider']
assert hashlib.sha256((provider/'qlib_store_manifest.json').read_bytes()).hexdigest()==plan['source_provider_manifest_sha256']
assert hashlib.sha256((root/plan['source_audit']).read_bytes()).hexdigest()==plan['source_audit_sha256']
parent=root/'research/minute_information_pit_allocator_20261012';summary=json.loads((parent/'summary.json').read_text())
assert summary['recorder_id']==plan['parent_training_source_recorder_id']
manifest=json.loads((parent/'verification_manifest.json').read_text())
for name,key in [('inputs/timing','parent_training_timing_artifact_sha256'),('inputs/features','parent_training_features_artifact_sha256')]:
    entry=next(x for x in manifest['entries'] if x['path'].endswith('/artifacts/'+name))
    assert entry['sha256']==plan[key] and hashlib.sha256((parent/entry['path']).read_bytes()).hexdigest()==entry['sha256']
qlib.init(provider_uri=str(provider),region='us',kernels=1)
calendar=pd.DatetimeIndex(D.calendar(start_time='2010-03-11',end_time='2026-09-25',freq='day'))
source=D.features(['SOXX'],['$close','$volume'],start_time='2010-03-11',end_time='2026-09-25',freq='day').xs('SOXX',level='instrument').sort_index()
assert source.index.equals(calendar) and source[['$close','$volume']].notna().all().all()
with R.uri_context(summary['recorder_uri']):
    rec=R.get_recorder(recorder_id=summary['recorder_id'],experiment_id=summary['experiment_id'])
    timing=rec.load_object('inputs/timing')
    existing=rec.load_object('inputs/features')
train_dates=pd.DatetimeIndex(timing.index)
train_dates=train_dates[(train_dates>=pd.Timestamp('2011-01-03')) & (train_dates<=pd.Timestamp('2022-12-30'))]
returns=source['$close'].pct_change()
sign=np.sign(returns)
vol=source['$volume'].astype(float)
weighted=(sign*vol).rolling(20,min_periods=20).sum()/vol.rolling(20,min_periods=20).sum()
unweighted=sign.rolling(20,min_periods=20).mean()
feature=(weighted-unweighted).replace([np.inf,-np.inf],np.nan)
train_feature=feature.reindex(train_dates)
assert len(train_dates)==3020
from zoneinfo import ZoneInfo
ny=ZoneInfo('America/New_York')
known=pd.DatetimeIndex([pd.Timestamp(day.date()).tz_localize(ny)+pd.Timedelta(hours=16,minutes=15) for day in train_dates])
decision=pd.to_datetime(timing.loc[train_dates,'decision_at'],utc=True)
assert (known.tz_convert('UTC')<pd.DatetimeIndex(decision)).all()
base=existing.reindex(train_dates)
assert len(base)==3020
correlations=base.apply(lambda s: train_feature.corr(s,method='spearman')).dropna().abs().sort_values(ascending=False)
control_corr=float(train_feature.corr(unweighted.reindex(train_dates),method='spearman'))
evaluation_dates=pd.DatetimeIndex(timing.index)
evaluation_dates=evaluation_dates[(evaluation_dates>=pd.Timestamp('2023-01-03')) & (evaluation_dates<=pd.Timestamp('2026-10-07'))]
missing_eval=evaluation_dates.difference(calendar)
result={'status':'training_source_coverage_passed_evaluation_tail_missing_no_fit' if train_feature.notna().all() else 'needs_training_source_history_no_fit',
 'source_calendar_first':str(calendar.min().date()),'source_calendar_last':str(calendar.max().date()),
 'training_signals':len(train_dates),'finite_candidate_training_signals':int(train_feature.notna().sum()),
 'evaluation_signal_dates_through_2026_10_07':len(evaluation_dates),'evaluation_source_dates_missing':len(missing_eval),
 'evaluation_missing_first':str(missing_eval.min().date()) if len(missing_eval) else None,
 'evaluation_missing_last':str(missing_eval.max().date()) if len(missing_eval) else None,
 'signed_volume_residual_min':float(train_feature.min()),'signed_volume_residual_max':float(train_feature.max()),
 'control_unweighted_sign_spearman':control_corr,
 'top_five_absolute_stage1_feature_spearman':{k:float(v) for k,v in correlations.head(5).items()},
 'max_absolute_stage1_feature_spearman':float(correlations.max()),
 'source_provider_manifest_sha256':plan['source_provider_manifest_sha256'],
 'source_audit_sha256':plan['source_audit_sha256'],'historical_live_receipt_certified':False,
 'evaluation_outcome_reads':0,'new_model_fits':0,'new_native_accounts':0,
 'next_action':'Qualify only the missing evaluation-tail SOXX daily volume source and verify its split adjustment and original receipt/clock. Freeze information gate and matched control before any Stage1 fit/account; reused market history remains development-only.'}
(card/'source_preflight_result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
print({k:result[k] for k in ['status','training_signals','finite_candidate_training_signals','evaluation_source_dates_missing','max_absolute_stage1_feature_spearman']})

{'status': 'training_source_coverage_passed_evaluation_tail_missing_no_fit', 'training_signals': 3020, 'finite_candidate_training_signals': 3020, 'evaluation_source_dates_missing': 8, 'max_absolute_stage1_feature_spearman': 0.24836803489276238}


## 八日原始日线补源与 Qlib 读回

按单次 Futu NONE/RTH 响应及独立的 corporate-action 查询证明 9/25–10/7 无新增事件，旧价格/成交量行严格不变，只补八个缺口。此步骤不读取评价收益或运行账户。

In [2]:
from src.data.qlib_native import build_qlib_store_from_frame
bridge=json.loads((card/'tail_bridge_plan.json').read_text())
assert hashlib.sha256((root/plan['source_audit']).read_bytes()).hexdigest()==bridge['parent_source_audit_sha256']
assert hashlib.sha256((provider/'qlib_store_manifest.json').read_bytes()).hexdigest()==bridge['parent_provider_manifest_sha256']
for name,digest in bridge['tail_input_sha256'].items():
    assert hashlib.sha256((card/name).read_bytes()).hexdigest()==digest
probe=json.loads((card/'futu_tail_probe_result.json').read_text())
rehab=json.loads((card/'futu_tail_rehab_result.json').read_text())
assert probe['status']=='eight_tail_rows_received_source_qualification_pending' and probe['rows']==8 and not probe['pagination_required']
assert rehab['status']=='tail_bridge_no_rehab_events' and rehab['window_event_rows']==0
raw=pd.read_csv(card/'futu_tail_soxx_none_raw.csv')
assert raw.code.eq('US.SOXX').all() and len(raw)==8
raw['datetime']=pd.to_datetime(raw.time_key).dt.normalize()
assert raw.datetime.dt.strftime('%Y-%m-%d').tolist()==bridge['expected_new_dates']
values=raw[['open','high','low','close','volume']].to_numpy(float)
assert np.isfinite(values).all() and (values>0).all()
assert ((raw.low<=raw.open)&(raw.open<=raw.high)&(raw.low<=raw.close)&(raw.close<=raw.high)).all()
parent_card=root/'research/canonical_price_basis_20261002'
parent_manifest=json.loads((parent_card/'artifact_manifest.json').read_text())
parent_entry=next(x for x in parent_manifest['entries'] if x['path'].endswith('/artifacts/data/qualified_frame.pkl'))
assert parent_entry['sha256']==bridge['parent_qualified_frame_artifact_sha256']
assert hashlib.sha256((root/parent_entry['path']).read_bytes()).hexdigest()==parent_entry['sha256']
with R.uri_context(str(parent_card/'qlib_recorder')):
    parent_rec=R.get_recorder(recorder_id=bridge['parent_qualified_price_recorder_id'],experiment_id='521389447037827398')
    qualified=parent_rec.load_object('data/qualified_frame.pkl')
last=qualified.loc[qualified.instrument.eq('SOXX')].sort_values('datetime').iloc[-1]
assert str(last.datetime.date())=='2026-09-25' and last.affine_A==1.0 and last.affine_B==0.0 and last.split_price_factor==1.0
assert np.isclose(last.close,float(source['$close'].iloc[-1]),rtol=1e-6,atol=1e-5)
assert np.isclose(last.volume,float(source['$volume'].iloc[-1]),rtol=1e-6,atol=1e-2)
old=source.rename(columns={'$close':'close','$volume':'volume'}).rename_axis('datetime').reset_index()
old['instrument']='SOXX'
new=raw[['datetime','close','volume']].copy();new['instrument']='SOXX'
combined=pd.concat([old[['datetime','instrument','close','volume']],new[['datetime','instrument','close','volume']]],ignore_index=True)
assert not combined.duplicated(['datetime','instrument']).any() and combined.datetime.is_monotonic_increasing
output=root/bridge['output_provider']
manifest=build_qlib_store_from_frame(combined,provider_uri=output,region='us',freq='day',as_of=bridge['as_of'],field_map={'close':'close','volume':'volume'},source='Parent published_futu_affine_20260925 Qlib values plus eight exact Futu NONE SOXX raw bars after no-rehab bridge; plan='+hashlib.sha256((card/'tail_bridge_plan.json').read_bytes()).hexdigest())
read=D.features(['SOXX'],['$close','$volume'],start_time='2010-03-11',end_time='2026-10-07',freq='day').xs('SOXX',level='instrument').sort_index()
assert len(read)==len(combined) and read.index.max()==pd.Timestamp('2026-10-07')
np.testing.assert_array_equal(read.loc[source.index,['$close','$volume']].to_numpy(),source[['$close','$volume']].to_numpy())
np.testing.assert_allclose(read.loc[new.datetime,['$close','$volume']].to_numpy(float),new[['close','volume']].to_numpy(float),rtol=1e-6,atol=1e-4)
ret=read['$close'].pct_change();sgn=np.sign(ret);vol=read['$volume'].astype(float)
pressure=((sgn*vol).rolling(20,min_periods=20).sum()/vol.rolling(20,min_periods=20).sum()-sgn.rolling(20,min_periods=20).mean()).replace([np.inf,-np.inf],np.nan)
evaluation_dates=pd.DatetimeIndex(timing.index)
evaluation_dates=evaluation_dates[(evaluation_dates>=pd.Timestamp('2023-01-03'))&(evaluation_dates<=pd.Timestamp('2026-10-07'))]
assert len(evaluation_dates)==944 and pressure.reindex(evaluation_dates).notna().all()
result={'status':'full_944_source_calendar_and_values_passed_information_unrun','tail_probe_plan_sha256':probe['plan_sha256'],'tail_probe_result_sha256':hashlib.sha256((card/'futu_tail_probe_result.json').read_bytes()).hexdigest(),'tail_rehab_result_sha256':hashlib.sha256((card/'futu_tail_rehab_result.json').read_bytes()).hexdigest(),'bridge_plan_sha256':hashlib.sha256((card/'tail_bridge_plan.json').read_bytes()).hexdigest(),'provider_manifest_sha256':hashlib.sha256((output/'qlib_store_manifest.json').read_bytes()).hexdigest(),'provider_uri':str(output),'calendar_first':str(read.index.min().date()),'calendar_last':str(read.index.max().date()),'source_rows':len(read),'new_tail_rows':8,'old_rows_exactly_unchanged':True,'new_raw_rows_qlib_readback_passed':True,'evaluation_signal_rows_with_finite_candidate':int(pressure.reindex(evaluation_dates).notna().sum()),'historical_original_receipt_certified':False,'market_requests_new_in_bridge':0,'model_fits':0,'native_accounts':0,'decision':'source_mechanics_passed_needs_frozen_information_test_and_forward_PIT'}
(card/'tail_bridge_result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
print({k:result[k] for k in ('status','source_rows','new_tail_rows','evaluation_signal_rows_with_finite_candidate','old_rows_exactly_unchanged')})


{'status': 'full_944_source_calendar_and_values_passed_information_unrun', 'source_rows': 4170, 'new_tail_rows': 8, 'evaluation_signal_rows_with_finite_candidate': 944, 'old_rows_exactly_unchanged': True}


## 中文结论

原 Stage1 的3020个训练日和944个评价信号日均有有限候选输入；训练域与39项原输入最大绝对秩相关0.248。旧Qlib行逐值不变，八条新增 Futu NONE 日线和无新增拆合股事件完成原生读回。来源包已归档NAS且远端哈希一致。此结果只证明来源与构造可行，不证明信息增量或净超额；历史实时首收、0bp账户、2/5bp和独立前向仍缺。下一步冻结信息门，未通过前不开户。
